# Experiment 4: Mean Filter With and Without Shared Memory 

Problem Statement


10.4 Experiment 4: Mean Filter With and Without Shared Memory Implement two versions of image blur: 1. Naive version where each thread reads its neighborhood from global memory. 2. Tiled version where each block loads a tile and halo region into shared memory.

The shared-memory version should reduce redundant global memory reads across neighboring threads.


## Understanding the Problem Statement

> What's a Mean Blur?

Think of it as the spatial cousin of your grayscale step: instead of mixing **R, G, B** at one location, you mix **nearby pixels** at one location.

Read the docs to understand how image kernels work with tiling.

Experiment Goals

- Create two kernels (one naive and the other with tiling and shared memory)
- Report and Analyze your observations from that

# Experiment

## Library Imports

In [4]:
!pip install ninja
!pip install pandas
!pip install matplotlib
!pip install numpy


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 301.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 177.9 MB/s eta 0:00:00
  Attempting uninstall: pyparsing
    Found existing installation: pyparsing 2.4.7
    Uninstalling pyparsing-2.4.7:
      Successfully uninstalled pyparsing-2.4.7

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [6]:
!nvidia-smi

Sun Jun 21 20:10:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 2000 Ada Gene...    On  |   00000000:82:00.0 Off |                  Off |
| 30%   27C    P8              7W /   70W |       4MiB /  16380MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path

LAB_DIR = Path.cwd() / "workspace"

assert torch.cuda.is_available(), "CUDA is required for this experiment"
device = torch.device("cuda")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {torch.cuda.get_device_name(device)}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Lab directory: {LAB_DIR}")


PyTorch: 2.4.1+cu124
Device: NVIDIA RTX 2000 Ada Generation
CUDA version: 12.4
Lab directory: /workspace


## Load the CUDA Extension

In [ ]:
from torch.utils.cpp_extension import load

image_blur_ext = load(
    name="image_blur_ext",
    sources=[
        str(LAB_DIR / "image_blur_ext.cpp"),
        str(LAB_DIR / "image_blur.cu")
    ],
    verbose=False
)

print("Loaded functions: ", [name for name in dir(image_blur_ext) if not name.startswith("_")])

In [ ]:
image_blur_ext.image_blur_naive(
    input_image:torch.Tensor,
    radius:int,
    threads_per_block:int
) -> torch.Tensor

image_blur_ext.image_blur_tiled(
    input_image:torch.Tensor,
    radius:int,
    threads_per_block:int,
    tile_size:int
) -> torch.Tensor

